In [ ]:
# @title اختار ملفات كونكر من Google Drive وارفعها إلى GitHub
# خلية واحدة: تصفح يدوي للمجلدات + اختيار ملف أو أكثر + رفعهم كـ Release Assets.
# لا يتم تعديل ملفات المشروع أو فرع main.

from google.colab import drive
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as W
import requests, hashlib, os
from urllib.parse import quote

REPO = "ZzwahwahzZ/Clash-of-Legends"
BRANCH = "tools/colab-drive-asset-transfer"
TAG = "original-mobile-assets-2026-11-06"

print("1) توصيل Google Drive...")
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
SHARED = Path("/content/drive/Shareddrives")
if not MYDRIVE.exists():
    raise FileNotFoundError("Google Drive/MyDrive غير ظاهر بعد التوصيل.")

archive_exts = {".zip", ".7z", ".rar", ".tar", ".gz", ".bz2", ".xz", ".pak", ".obb"}
current_dir = MYDRIVE

root_picker = W.Dropdown(
    options=[("My Drive", str(MYDRIVE))] +
            ([("Shared drives", str(SHARED))] if SHARED.exists() else []),
    value=str(MYDRIVE), description="مكان البحث:", layout=W.Layout(width="98%")
)
folder_picker = W.Dropdown(options=[], description="المجلد:", layout=W.Layout(width="98%"))
files_picker = W.SelectMultiple(
    options=[], description="الملفات:", rows=12,
    layout=W.Layout(width="98%", height="270px")
)
show_all = W.Checkbox(value=False, description="إظهار كل أنواع الملفات (مش الأرشيفات فقط)")
up_btn = W.Button(description="⬆ مجلد لأعلى", button_style="")
open_btn = W.Button(description="فتح المجلد المحدد", button_style="info")
refresh_btn = W.Button(description="تحديث القائمة")
token_box = W.Password(
    description="GitHub Token:", placeholder="الصق التوكن هنا — لن يظهر على الشاشة",
    layout=W.Layout(width="98%")
)
upload_btn = W.Button(description="رفع الملفات التي اخترتها", button_style="success", icon="upload")
path_label = W.HTML()
status = W.Output()

def readable_size(n):
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.1f} {unit}"
        x /= 1024

def refresh_browser(*_):
    global current_dir
    path_label.value = f"<b>المسار الحالي:</b> {current_dir}"
    dirs, files = [], []
    try:
        with os.scandir(current_dir) as it:
            for entry in it:
                try:
                    if entry.is_dir(follow_symlinks=False):
                        dirs.append((entry.name, entry.path))
                    elif entry.is_file(follow_symlinks=True):
                        p = Path(entry.path)
                        if show_all.value or p.suffix.lower() in archive_exts:
                            files.append((f'{entry.name}  —  {readable_size(entry.stat().st_size)}', entry.path))
                except (PermissionError, OSError):
                    continue
    except Exception as e:
        with status:
            print(f"تعذر قراءة المجلد: {e}")
        return
    folder_picker.options = sorted(dirs, key=lambda x: x[0].lower())
    files_picker.options = sorted(files, key=lambda x: x[0].lower())
    files_picker.value = ()
    with status:
        clear_output()
        print(f"المجلدات: {len(dirs)} | الملفات المعروضة: {len(files)}")
        print("اختار مجلدًا وافتحه، أو علّم ملفًا/أكثر من قائمة الملفات.")

def change_root(change):
    global current_dir
    if change["name"] == "value":
        current_dir = Path(change["new"])
        refresh_browser()

def go_up(_):
    global current_dir
    base = Path(root_picker.value)
    if current_dir != base and base in current_dir.parents:
        current_dir = current_dir.parent
        refresh_browser()

def open_folder(_):
    global current_dir
    if folder_picker.value:
        current_dir = Path(folder_picker.value)
        refresh_browser()

root_picker.observe(change_root, names="value")
up_btn.on_click(go_up)
open_btn.on_click(open_folder)
refresh_btn.on_click(refresh_browser)
show_all.observe(refresh_browser, names="value")

def upload_selected(_):
    with status:
        clear_output()
        selected = list(files_picker.value)
        if not selected:
            print("اختار ملفًا واحدًا على الأقل أولًا.")
            return
        token = token_box.value.strip()
        if not token:
            print("اكتب GitHub Token في الخانة المخصصة.")
            return

        api = "https://api.github.com"
        headers = {
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28"
        }
        print(f"تم اختيار {len(selected)} ملف/ملفات.")
        check = requests.get(f"{api}/repos/{REPO}", headers=headers, timeout=30)
        check.raise_for_status()
        if not check.json().get("permissions", {}).get("push", False):
            raise PermissionError("التوكن لا يملك صلاحية الكتابة على المستودع.")

        rel = requests.get(f"{api}/repos/{REPO}/releases/tags/{TAG}", headers=headers, timeout=30)
        if rel.status_code == 200:
            release = rel.json()
        elif rel.status_code == 404:
            made = requests.post(
                f"{api}/repos/{REPO}/releases",
                headers=headers,
                json={
                    "tag_name": TAG,
                    "target_commitish": BRANCH,
                    "name": "Original Conquer Mobile Resource Files",
                    "body": "Original Conquer Mobile files selected manually from the user's Google Drive for asset forensics.",
                    "draft": True,
                    "prerelease": True
                },
                timeout=60
            )
            made.raise_for_status()
            release = made.json()
        else:
            rel.raise_for_status()

        existing = {a["name"]: a for a in release.get("assets", [])}
        results = []
        for item in selected:
            source = Path(item)
            size = source.stat().st_size
            if size == 0:
                print(f"تخطي ملف فارغ: {source.name}")
                continue
            if source.name in existing:
                old = existing[source.name]
                if old.get("size") == size:
                    print(f"موجود بالفعل بالحجم نفسه، لم أكرر رفعه: {source.name}")
                    results.append((source.name, old["browser_download_url"], size, "already existed"))
                    continue
                raise RuntimeError(
                    f"يوجد Release asset باسم {source.name} لكن حجمه مختلف. "
                    "لم أستبدله حفاظًا على النسخة الموجودة. غيّر اسم الملف محليًا أو راجع النسخة الحالية."
                )

            print(f"\nتجهيز {source.name} — {readable_size(size)}")
            h = hashlib.sha256()
            with source.open("rb") as f:
                while True:
                    block = f.read(16 * 1024 * 1024)
                    if not block:
                        break
                    h.update(block)
            sha = h.hexdigest()
            upload_url = release["upload_url"].split("{", 1)[0] + "?name=" + quote(source.name)
            up_headers = {**headers, "Content-Type": "application/octet-stream", "Content-Length": str(size)}
            print("SHA-256:", sha)
            print("جارٍ الرفع — اترك جلسة Colab مفتوحة...")
            with source.open("rb") as f:
                response = requests.post(
                    upload_url, headers=up_headers, data=f, timeout=(60, 7200)
                )
            response.raise_for_status()
            asset = response.json()
            if asset.get("name") != source.name or asset.get("size") != size:
                raise RuntimeError(f"بيانات الملف المرفوع لا تطابق المصدر: {source.name}")
            results.append((source.name, asset["browser_download_url"], size, sha))
            existing[source.name] = asset
            print("اكتمل رفع:", source.name)

        print("\n========== تقرير النقل ==========")
        print("Release:", release.get("html_url"))
        for name, url, size, fingerprint in results:
            print(f"\nالملف: {name}")
            print("الحجم:", f"{size:,}", "bytes")
            print("الرابط:", url)
            print("SHA-256 / الحالة:", fingerprint)
        print("\nانسخ التقرير كاملًا وأرسله لي هنا.")
        token_box.value = ""

upload_btn.on_click(upload_selected)
display(W.VBox([
    W.HTML("<h3>اختيار ملفات كونكر من Google Drive</h3><p>تنقّل بين المجلدات، ثم حدّد الملفين أو أي ملفات أخرى تريد إرسالها.</p>"),
    root_picker, path_label, W.HBox([up_btn, open_btn, refresh_btn]),
    folder_picker, show_all, files_picker,
    token_box, upload_btn, status
]))
refresh_browser()
